In [ ]:
from openai import OpenAI
from pdf2image import convert_from_path
from pydantic import ConfigDict, BaseModel, Field
from typing import List
from pathlib import Path
import tiktoken
import base64
import json
import re

#client = OpenAI(api_key="your_api_key_here")
client = OpenAI()

In [ ]:
pdf = Path("example.pdf") # input PDF
out = Path(pdf.stem) # output folder
out.mkdir(parents=True, exist_ok=True) # create folder if it does not exist

images = convert_from_path(pdf, output_folder=out, fmt="png", output_file=pdf.stem)

In [ ]:
# Image extraction prompt
prompt = """You are a professional OCR engine. Transcribe the text EXACTLY as written.
Do not correct spelling or grammar errors.
If text is blurry use context to infer the word.
Ignore headers footers as well as line and page numbers.
"""

pages = []
for image in images:
    with open(image.filename, "rb") as f:
         base64_image = base64.b64encode(f.read()).decode()

    # Image extraction call
    response = client.responses.create(
        model="gpt-4.1-mini-2025-04-14",
        temperature = 0.0,
        input=[
            {
                "role": "user",
                "content": [
                    { "type": "input_text", "text": prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                    },
                ],
            }
        ],
    )
    pages.append(response.output_text)

raw_text = "\n\n".join(pages)
Path(f"{pdf.stem}.txt").write_text(raw_text, encoding="utf-8")

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")


def chunk_on_colon_boundary(text: str, max_tokens: int = 10000) -> list[str]:
    tokens = enc.encode(text)
    chunks = []
    start = 0
    L = len(tokens)

    while start < L:
        end = min(start + max_tokens, L)
        chunk = enc.decode(tokens[start:end])
        remaining = enc.decode(tokens[end:])
        match = re.search(
            r"(?:^|\n)(?:Dr|Mr|Mrs|Ms|Prof|Chair|Lt)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*:",
            remaining,
            re.IGNORECASE,
        )
        if match:
            colon_idx = match.start()
            extra = remaining[:colon_idx]
            combined = chunk + extra
            end = start + len(enc.encode(combined))
        else:
            combined = chunk
        chunks.append(combined)
        start = end

    return chunks


chunk_size = 5000
chunks = chunk_on_colon_boundary(text=raw_text, max_tokens=chunk_size)

In [ ]:
# Pydantic class for speaker-statement pairs
class TranscriptEntry(BaseModel):
    model_config = ConfigDict(extra="forbid")

    speaker: str = Field(
        description="The speaker's name in UPPERCASE. Use 'UNKNOWN' if the text starts without a name. Do not extract names from headers or rosters."
    )
    statement: str = Field(
        description="The spoken text verbatim, including all bracketed [...] or parenthetical (...) notes like [Laughter.], (Applause.). Remove all line breaks (\\n) so it is a single line."
    )

# Pydantic schema class
class TranscriptResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")

    entries: List[TranscriptEntry] = Field(
        description="A list of transcript entries."
    )

In [ ]:
# Text structuring prompt
system_prompt = """You are an expert transcript cleaner. You will receive a raw text chunk. Your goal is to extract the dialogue into a structured format.

**PRIME DIRECTIVE - WORD FIDELITY VS. FORMAT FLEXIBILITY:**
1. **DO NOT** change words, fix grammar,  remove non-verbal markers (e.g., [Laughter.], (Applause.)) or autocomplete sentences.
2. **DO** fix whitespace. You must remove line breaks within a sentence and normalize multiple spaces into a single space.

Follow these processing rules:

1. **Scope & Exclusions:**
   - **Ignore Metadata:** Do not extract meeting rosters, attendee lists, or headers. Start extracting only when the actual dialogue begins.
   - **Ignore Artifacts:** Remove page numbers, file paths, and margin line numbers.

2. **Speaker Identification:**
   - **Standard Speech:** Extract the speaker's name exactly as written and convert to UPPERCASE.
   - **Orphaned Speech:** If the chunk starts with sentences/dialogue but has no speaker name attached, label the speaker as: UNKNOWN."""

entries = []
for chunk in chunks:
    # text structuring call
    response = client.responses.parse(
        model="gpt-4.1-mini-2025-04-14",
        temperature=0.0,
        input=[
            {"role": "system", "content": system_prompt},  # Prompt instructions
            {"role": "user", "content": chunk}             # Section of raw text
        ],
        text_format=TranscriptResponse
    )
    entries.extend(response.output_parsed.entries)

In [ ]:
with open(f"{pdf.stem}.json", "w", encoding="utf-8") as f:
    json.dump([entry.model_dump() for entry in entries], f, ensure_ascii=False, indent=2)